# Part 1 — Data Preparation


### Part 1.1 Load data and Inspect

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("COVID-19BehaviorData_CAN2022.csv", low_memory=False)

print(df_raw.shape)
display(df_raw[["i9_health", "age", "cantril_ladder", "region"]].head())

### Part 1.2 Check missing values

The initial check counts blank or missing entries in the selected columns. After selecting valid target responses, we also treat "Don't know" and "Prefer not to say" as missing. This is why the later missing-value counts are higher.

### Part 1.3 Select our target (i9_health) and predictors (15 vars)

We predict whether respondents intend to self-isolate when unwell. The target is `i9_health` (Yes or No), with "No" treated as the positive class. Identifying respondents less willing to self-isolate could help the Public Health Agency of Canada plan targeted communication. We use demographic, wellbeing, vaccination-attitude, and preventive-behavior predictors.

In [ ]:
target = "i9_health"

num_cols = [
    "age",
    "cantril_ladder"
]

cat_cols = [
    "gender",
    "region",
    "household_size",
    "household_children",
    "employment_status",
    "qweek",
    "PHQ4_1",
    "PHQ4_3",
    "vac2_1",
    "vac2_2",
    "vac7",
    "WCRex1",
    "Vent_3"
]

features = num_cols + cat_cols

df_clean = df_raw[[target] + features].copy()

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].mask(df_clean[col].str.fullmatch(r"\s*").fillna(False))

for col in num_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
)

print("Selected predictors:", len(features))
display(df_clean.head())

missing = df_clean.isna().mean().mul(100).sort_values(ascending=False)
display(missing[missing > 0].round(1))

### Part 1.4 For Data Encoding & Scaling rules future Modeling
- Age, Cantril Ladder, Household, qweek: Numerical
- Gender: Binary
- Region, Employment: One-Hot Encoding
- PHQ4, Vac2, WCRex1: Ordinal Encoding

In [ ]:
model_df = df_clean[
    df_clean["i9_health"].isin(["Yes", "No"])
].copy()

model_df = model_df.replace(["Prefer not to say", "Don't know"], np.nan)

model_df.isna().sum()

Convert text-coded numerical variables to numbers:

In [ ]:
model_df["target"] = model_df["i9_health"].map({"Yes": 0, "No": 1})
model_df["gender"] = model_df["gender"].map({"Female": 0, "Male": 1})

model_df["qweek"] = model_df["qweek"].str.removeprefix("week ").astype(int)

model_df["vac2_1"] = model_df["vac2_1"].str[:1].astype(int)
model_df["vac2_2"] = model_df["vac2_2"].str[:1].astype(int)

model_df["household_children"] = pd.to_numeric(
    model_df["household_children"].str.extract(r"(\d+)")[0],
    errors="coerce"
)
model_df["household_size"] = pd.to_numeric(
    model_df["household_size"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

### Part 1.5 Train/Test Split in case for "Test Set Leakage"

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df,
    test_size = 0.2,
    random_state = 5,
    stratify=model_df["target"]
)

X_train = train_df[features].copy()
X_test = test_df[features].copy()

y_train = train_df["target"]
y_test = test_df["target"]

print("Training set:", train_df.shape)
print("Test set:", test_df.shape)

print("\nTraining target distribution:")
print(train_df["i9_health"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df["i9_health"].value_counts(normalize=True))

Use ordinal encoding (OrdinalEncoder) to transform ordinal variables:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

wellbeing_levels = ["Not at all", "Several days", "More than half the days", "Nearly every day"]
phq4_encoder = OrdinalEncoder(categories=[wellbeing_levels, wellbeing_levels],
                              handle_unknown="use_encoded_value", 
                              unknown_value=np.nan)
X_train[["PHQ4_1", "PHQ4_3"]] = phq4_encoder.fit_transform(X_train[["PHQ4_1", "PHQ4_3"]])
X_test[["PHQ4_1", "PHQ4_3"]] = phq4_encoder.transform(X_test[["PHQ4_1", "PHQ4_3"]])


vac_trust_levels = ["Not at all", "A little", "Moderately", "Very much"]
vac7_encoder = OrdinalEncoder(categories=[vac_trust_levels],
                              handle_unknown="use_encoded_value", 
                              unknown_value=np.nan)
X_train[["vac7"]] = vac7_encoder.fit_transform(X_train[["vac7"]])
X_test[["vac7"]] = vac7_encoder.transform(X_test[["vac7"]])

gov_eval_levels = ["Very badly", "Somewhat badly", "Somewhat well", "Very well"]
wcrex1_encoder = OrdinalEncoder(categories=[gov_eval_levels],
                                handle_unknown="use_encoded_value", 
                                unknown_value=np.nan)
X_train[["WCRex1"]] = wcrex1_encoder.fit_transform(X_train[["WCRex1"]])
X_test[["WCRex1"]] = wcrex1_encoder.transform(X_test[["WCRex1"]])

outdoor_freq_levels = ["Not at all", "Rarely", "Sometimes", "Frequently", "Always"]
vent3_encoder = OrdinalEncoder(categories=[outdoor_freq_levels],
                               handle_unknown="use_encoded_value", 
                               unknown_value=np.nan)
X_train[["Vent_3"]] = vent3_encoder.fit_transform(X_train[["Vent_3"]])
X_test[["Vent_3"]] = vent3_encoder.transform(X_test[["Vent_3"]])

Use one-hot encoding (OneHotEncoder) for categorical variables:

In [ ]:
from sklearn.preprocessing import OneHotEncoder
onehot_cols = ["region", "employment_status"]
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_encoded = pd.DataFrame(
    enc.fit_transform(X_train[onehot_cols]),
    columns=enc.get_feature_names_out(onehot_cols),
    index=X_train.index
)
test_encoded = pd.DataFrame(
    enc.transform(X_test[onehot_cols]),
    columns=enc.get_feature_names_out(onehot_cols),
    index=X_test.index
)

X_train = pd.concat([X_train.drop(columns=onehot_cols), train_encoded], axis=1)
X_test = pd.concat([X_test.drop(columns=onehot_cols), test_encoded], axis=1)

# Part 2 — Exploratory Data Analysis


### Part 2.1 Target Distribution
    Purpose: Checking if there is any Class Imbalance for Training Set and Target Distribution

In [ ]:
import matplotlib.pyplot as plt

print(train_df["i9_health"].value_counts())

print(train_df["i9_health"].value_counts(normalize=True) * 100)

train_df["i9_health"].value_counts().plot(kind="bar")

plt.title("Self-Isolation Intention (Training Data)")
plt.xlabel("Response")
plt.ylabel("Number of Respondents")
plt.xticks(rotation=0)
plt.show()

### Part 2.2 Region & Target Summary
    Purpose: Sample size Visualization & Self-Isolation Intention by region

In [ ]:
region_summary = pd.crosstab(
    train_df["region"],
    train_df["i9_health"]
)

region_summary["Total"] = region_summary.sum(axis=1)
region_summary["No Rate"] = (
    region_summary["No"] / region_summary["Total"] * 100
)

display(region_summary.sort_values("No Rate", ascending=False))

### Part 2.3 Categorical variable Vs. Target 
#### Government Performance vs Self-Isolation Intention
    Purpose: Examine whether respondents' evaluations of government pandemic management are associated with self-isolation intention.

Respondents rating government performance "Very badly" have a higher "No" rate (about 31%) than those rating it "Very well" (about 16%). These groups may differ in other ways, so this does not establish causation.

In [ ]:
government_pct = pd.crosstab(
    train_df["WCRex1"],
    train_df["i9_health"],
    normalize="index"
)

display(government_pct)

government_pct.plot(kind="bar", stacked=True)

plt.title("Self-Isolation Intention by Government Evaluation")
plt.xlabel("Government Evaluation")
plt.ylabel("Proportion")
plt.xticks(rotation=45, ha="right")
plt.show()

#### Social Contact Avoidance vs Self-Isolation Intention
    Purpose: Examine how recent preventive behavior relates to willingness to self-isolate.

Respondents reporting "Not at all" for social contact avoidance have a higher "No" rate (about 40%) than those reporting "Always" (about 16%). This is a descriptive association, not a causal effect.

In [ ]:
vent_pct = pd.crosstab(
    train_df["Vent_3"],
    train_df["i9_health"],
    normalize="index"
)

display(vent_pct)

vent_pct.plot(kind="bar", stacked=True)

plt.title("Self-Isolation Intention by Social Contact Avoidance")
plt.xlabel("Social Contact Avoidance")
plt.ylabel("Proportion")
plt.xticks(rotation=45, ha="right")
plt.show()

#### Survey Week vs Self-Isolation Intention
    Purpose: Investigate whether self-isolation intentions differ across survey weeks.

The share answering "No" rises from about 15% in week 44 to about 30% in week 50. This is an association across survey weeks, not evidence that time itself caused the change.

In [ ]:
print(train_df["qweek"].value_counts().sort_index())

week_pct = pd.crosstab(
    train_df["qweek"],
    train_df["i9_health"],
    normalize="index"
)

display(week_pct)

week_pct["No"].plot(kind="line", marker="o")

plt.title("Unwillingness to Self-Isolate by Survey Week")
plt.xlabel("Survey Week")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Proportion Answering No")
plt.show()



### Part 2.4 Numeric Distributions
    Purpose: Checking the distribution for numerical variables to find any outliers or skews

In [ ]:
print(train_df[num_cols].describe())

train_df[num_cols].hist(
    bins=20,
    figsize=(9, 4)
)

plt.show()

train_df[num_cols].boxplot(figsize=(7, 4))

plt.title("Numerical Predictor Distributions")
plt.show()

### Part 2.5 Numeric Predictors Vs. Target
    Purpose: Checking if there is diff age distribution within Target variable

In [ ]:
train_df.boxplot(
    column="age",
    by="i9_health"
)

plt.title("Age by Self-Isolation Intention")
plt.suptitle("")
plt.xlabel("Response")
plt.ylabel("Age")
plt.show()

In [ ]:
train_df = train_df.copy()

train_df["age_group"] = pd.cut(
    train_df["age"],
    bins=[17, 29, 44, 59, 120],
    labels=["18-29", "30-44", "45-59", "60+"]
)

age_pct = pd.crosstab(
    train_df["age_group"],
    train_df["i9_health"],
    normalize="index"
)

print(age_pct)

age_pct.plot(kind="bar", stacked=True)

plt.title("Self-Isolation Intention by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Proportion")
plt.xticks(rotation=0)
plt.show()

### Part 2.6 Multivariate Analysis
#### Age x Life satisfaction -> No Rate
    Purpose: Investigate the relationship b/w Age & Life Satisfaction after the combination

In [ ]:
import seaborn as sns

sns.scatterplot(
    data=train_df,
    x="age",
    y="cantril_ladder",
    hue="i9_health",
    alpha=0.4
)

plt.title("Age vs Life Satisfaction by Isolation Intention")
plt.show()

The scatter plot examines the relationship between age, life satisfaction, and self-isolation intention.

There is substantial overlap between respondents who answered "Yes" and "No" across different ages and life satisfaction scores. No clear separation between the two groups is visible.

This suggests that age and life satisfaction alone may not be sufficient to distinguish respondents' self-isolation intentions. However, overlapping points make it difficult to assess the strength of their relationship visually.

In [ ]:
train_df["satisfaction_group"] = pd.cut(
    train_df["cantril_ladder"],
    bins=[-1, 3, 6, 10],
    labels=["Low (0-3)", "Medium (4-6)", "High (7-10)"]
)

heatmap_data = train_df.pivot_table(
    index="age_group",
    columns="satisfaction_group",
    values="target",
    aggfunc="mean",
    observed=True
)

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1%",
    cmap="Oranges"
)

plt.title("No Rate by Age and Life Satisfaction")
plt.xlabel("Life Satisfaction")
plt.ylabel("Age Group")
plt.show()

#### Government Evaluation × Social Contact Avoidance → No Rate
    Purpose: Explore whether self-isolation intention varies across combinations of government evaluation and preventive behavior.

In [ ]:
multi = train_df.pivot_table(
    index="WCRex1",
    columns="Vent_3",
    values="target",
    aggfunc="mean"
)

sns.heatmap(
    multi,
    annot=True,
    fmt=".1%",
    cmap="Oranges"
)

plt.title("No Rate by Government Evaluation and Social Contact Avoidance")
plt.show()

In [ ]:
pd.crosstab(
    train_df["WCRex1"],
    train_df["Vent_3"]
)

### Part 2.7 Correlation Analysis
    Purpose: Checking any correlation and duplicate information b/w numerical variables

In [ ]:
corr = train_df[num_cols].corr()

print(corr)

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation Matrix")
plt.show()

# Part 3 — KNN Modeling


### Part 3.1 Handle missing values within cross-validation

Missing values are imputed separately within each training fold to avoid using validation-fold information.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

mode_cols = ["PHQ4_1", "PHQ4_3", "WCRex1"]
median_cols = ["household_size", "household_children"]

imputer = ColumnTransformer([
    ("mode", SimpleImputer(strategy="most_frequent"), mode_cols),
    ("median", SimpleImputer(strategy="median"), median_cols)
], remainder="passthrough")

### Part 3.2 KNN model and Hyperparameter Tuning
We tune the model on both $k$ and voting method (uniform or distance).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

seed = 5
scoring = "neg_mean_squared_error"
num_folds = 10

model = Pipeline([
    ("imputer", imputer),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

kfold = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)

param_grid = {
    "knn__n_neighbors": np.arange(1, 20),
    "knn__weights": ["uniform", "distance"]
}

grid = GridSearchCV(model, param_grid=param_grid, scoring=scoring, cv=kfold)
grid_result = grid.fit(X_train, y_train)

In [ ]:
grid_df = pd.DataFrame(grid_result.cv_results_["params"])
grid_df["mean_test_score"] = grid_result.cv_results_["mean_test_score"]

sns.lineplot(
    data=grid_df,
    x="knn__n_neighbors",
    y="mean_test_score",
    hue="knn__weights"
)

plt.xticks(np.arange(1, 20))
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Negative Mean Squared Error")
plt.title("KNN Cross-Validation Performance")
plt.show()

Smaller values of k can overfit local noise, while very large values can smooth away useful predictor information. We compare k and voting weights using stratified 10-fold cross-validation, selecting the combination with the lowest average validation error.

In [ ]:
best_k = grid_result.best_params_["knn__n_neighbors"]
best_weights = grid_result.best_params_["knn__weights"]
cv_error = -grid_result.best_score_

print(f"Best model: k = {best_k}, weights = {best_weights}")
print(f"10-fold CV misclassification rate: {cv_error:.1%}")

# Part 4 — Model Evaluation and Performance Metrics
## 4.1 Test Set Predictions

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

best_model = grid_result.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1 Score:", round(f1, 3))
print("ROC AUC:", round(auc, 3))

## 4.2 Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Yes", "No"]
)

disp.plot(cmap="Blues")
plt.title("KNN Confusion Matrix")
plt.show()

## 4.3 ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"KNN (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("KNN ROC Curve")
plt.legend()
plt.show()

## 4.4 Performance Metrics Summary

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"],
    "Score": [accuracy, precision, recall, f1, auc]
})

metrics_df["Score"] = metrics_df["Score"].round(3)

display(metrics_df)

In [ ]:
from IPython.display import Markdown

baseline_accuracy = (y_test == 0).mean()
tn, fp, fn, tp = cm.ravel()

display(Markdown(
    f"The KNN model achieved **{accuracy:.1%} accuracy**, **{precision:.1%} precision**, "
    f"**{recall:.1%} recall**, **{f1:.1%} F1-score**, and **{auc:.3f} ROC AUC** "
    f"on the held-out test set. Predicting 'Yes' for everyone would achieve "
    f"**{baseline_accuracy:.1%} accuracy**, so accuracy alone is misleading.\n\n"
    f"The confusion matrix shows that the model correctly identified **{tp} of {tp + fn}** "
    f"respondents answering 'No' and missed **{fn}**. For the Public Health Agency "
    "of Canada, recall for the 'No' class is especially important because missed "
    "respondents may not receive targeted public-health messaging. The ROC AUC "
    "indicates some ability to distinguish the groups across thresholds, but low "
    "recall limits the model's usefulness at the current threshold. Future work "
    "could explore class imbalance and threshold selection. These are predictions "
    "of self-reported intentions, not actual future behavior."
))

## 4.5 Model Performance Interpretation

The paragraph above summarizes the test-set results and their implications for targeted public-health communication.